# **Atividade Prática**
<font size=3>

- **Tema:** Vetores de frequência *vs* semânticos em modelos de aprendizado de máquina.
- **Prazo de entrega:** 20 de Abril.

**Envie** o notebook **executado** em formato **ipynb** pelo [formulário](https://docs.google.com/forms/d/e/1FAIpQLSfhkf8HoNNsr9WixEVVlxh8-pFK-rnXsLKN_OLRH_Tg5-5SmA/viewform?usp=sharing&ouid=111377632325147218671).

---

<font size=3>

## **Enunciado:**
Nessa atividade, iremos utilizar o [conjunto de dados](https://huggingface.co/datasets/Yelp/yelp_review_full) da plataforma Yelp, onde temos **resenhas sobre estabelecimentos** como restaurantes, bares, lojas, salões de beleza, mecânicas etc. As resenhas estão **ranqueadas de 1 a 5 estrelas**, com base na experiência de diferentes pessoas. Os dados consistem 650.000 resenhas, mas por fins de consumo de memória e tempo de execução do modelo, iremos utilizar apenas 5.000 resenhas &mdash; que estão diponíveis no diretório **dataset** deste *drive*.

---

### **1. Escolha de vetorização:**
<font size=3>

Com base no *pipeline* (apresentado no **_Notebook_ 6**), resolva a **tarefa de classificação** do conjunto de dados da plataforma **Yelp** com o modelo **k-vizinhos mais próximos** ([k-NN](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)). **Compare** a performance modelo com base na vetorização [BoW](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) e a vetorização [TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).
   

#### Imports e configurações globais

In [1]:
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, normalize

Abaixo os parametros gerais aplicáveis a todos os vetorizadores e modelos (se aplicável). Na eventualidade de uso de parametros especificos para um tipo específico de vetorização e/ou modelo, tais parâmetros serão definidos na seção correspondente. Aqui somente os parametros gerais que aplicam a tudo.

In [2]:
MIN_DF=5 # mínimo de documentos que uma palavra deve aparecer para ser considerada
MAX_DF=0.95 # máximo de documentos que uma palavra deve aparecer para ser considerada
RANDOM_STATE=42 # semente para garantir a reprodutibilidade dos resultados
TEST_SIZE=0.10 # proporção de dados para teste
SVD_COMPONENTS=25 # número de componentes para o SVD
#### **1.1 Vetorização com BoW:**

In [3]:
df = pd.read_csv("./dataset/yelp_reviews.csv")

print(df.shape)

df.head()

(5000, 2)


,text,label
0,dr. goldberg offers everything i look for in a...,4
1,"Unfortunately, the frustration of being Dr. Go...",1
2,Been going to Dr. Goldberg for over 10 years. ...,3
3,Got a letter in the mail last week that said D...,3
4,I don't know what Dr. Goldberg was like before...,0


In [4]:
y = df["label"].to_numpy()
texts = df["text"].tolist()

texts[1]

"Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff.  It seems that his staff simply never answers the phone.  It usually takes 2 hours of repeated calling to get an answer.  Who has time for that or wants to deal with it?  I have run into this problem with many other doctors and I just don't get it.  You have office workers, you have patients with medical needs, why isn't anyone answering the phone?  It's incomprehensible and not work the aggravation.  It's with regret that I feel that I have to give Dr. Goldberg 2 stars."

In [5]:
# Faz o download da lista de stop words:
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gilcesarf/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
lemmatizer = WordNetLemmatizer()

def Preprocessor(text, lemma=False):

    text = text.lower() # converte para minúsculas

    text = re.sub(r"[^\w]", " ", text) # substitui caracteres não alfanuméricos por espaços
    text = re.sub(r"\s+", " ", text) # substitui múltiplos espaços por um único espaço
    text = re.sub(r"\d+", "", text) # remove dígitos
    text = text.strip() # remove espaços em branco no início e no final

    if lemma:
        text = " ".join([lemmatizer.lemmatize(token) for token in text.split()]) # aplica lematização

    return " ".join([token for token in text.split() if token not in stop_words]) # remove stop words


Preprocessor(texts[1], lemma=False)

'unfortunately frustration dr goldberg patient repeat experience many doctors nyc good doctor terrible staff seems staff simply never answers phone usually takes hours repeated calling get answer time wants deal run problem many doctors get office workers patients medical needs anyone answering phone incomprehensible work aggravation regret feel give dr goldberg stars'

#### **1.1 Vetorização com BoW:**

In [7]:
vec_bow = CountVectorizer(preprocessor=Preprocessor, min_df=MIN_DF, max_df=MAX_DF)

X_bow = vec_bow.fit_transform(texts)

vocab = vec_bow.get_feature_names_out()

print(f"Tamanho do vocabulário BoW: {len(vocab)}")

print(f"Primeiros 10 tokens: {vocab[:10]}")
print(f"Últimos 10 tokens: {vocab[-10:]}")
print(f"Dimensão da matriz documento-termo com BoW: {X_bow.toarray().shape}")
print(f"Objeto vetorizador BoW:")
display(vec_bow)

Tamanho do vocabulário BoW: 5859
Primeiros 10 tokens: ['abandoned' 'ability' 'able' 'absence' 'absolute' 'absolutely' 'absurd'
 'abundance' 'abysmal' 'ac']
Últimos 10 tokens: ['yuck' 'yuengling' 'yuk' 'yum' 'yummy' 'yup' 'zero' 'zinho' 'zone'
 'zucchini']
Dimensão da matriz documento-termo com BoW: (5000, 5859)
Objeto vetorizador BoW:


,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",<function Pre...t 0x11772b920>
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [8]:
print(f"Frequência de classes: {np.unique(y, return_counts=True)}")

Frequência de classes: (array([0, 1, 2, 3, 4]), array([ 883, 1126, 1119,  978,  894]))


Podemos verificar que existem 5 classes, numeradas de 0 a 4, e a distribuição entre cada classe sugere que os experimentos podem se beneficiar se aplicarmos estratificação nos algoritmos de separação de dados de treino e teste.

In [9]:
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

def run_train_predict_model(
    X,
    y,
    model=None,
    scaling='standard'  # None | "standard" | "normalize"
):

    if model is None:
        model = KNeighborsClassifier(n_neighbors=5)

    # valida antes de qualquer transformação
    if hasattr(model, "metric") and model.metric == "cosine" and scaling == "standard":
        raise ValueError("Não use StandardScaler com cosine. Use scaling='normalize'.")

    if scaling not in (None, "standard", "normalize"):
        raise ValueError("scaling deve ser None, 'standard' ou 'normalize'")
        
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE
    )

    # 🔹 pré-processamento
    if scaling == "standard":
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    elif scaling == "normalize":
        X_train = normalize(X_train)
        X_test = normalize(X_test)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"Acc = {acc:.3f}")

    return y_pred, acc

In [10]:
y_pred_bow_raw, acc_bow_raw = run_train_predict_model(X_bow.toarray(), y, scaling='standard')


Acc = 0.220


#### **1.2 Vetorização TF-IDF:**

In [11]:
vec_tfidf = TfidfVectorizer(preprocessor=Preprocessor, min_df=MIN_DF, max_df=MAX_DF)

X_tfidf = vec_tfidf.fit_transform(texts)

vocab_tfidf = vec_tfidf.get_feature_names_out()

print(f"Tamanho do vocabulário TF-IDF: {len(vocab_tfidf)}")

print(f"Primeiros 10 tokens: {vocab_tfidf[:10]}")
print(f"Últimos 10 tokens: {vocab_tfidf[-10:]}")
print(f"Dimensão da matriz documento-termo com TF-IDF: {X_tfidf.toarray().shape}")
print(f"Objeto vetorizador TF-IDF:")
display(vec_tfidf)

Tamanho do vocabulário TF-IDF: 5859
Primeiros 10 tokens: ['abandoned' 'ability' 'able' 'absence' 'absolute' 'absolutely' 'absurd'
 'abundance' 'abysmal' 'ac']
Últimos 10 tokens: ['yuck' 'yuengling' 'yuk' 'yum' 'yummy' 'yup' 'zero' 'zinho' 'zone'
 'zucchini']
Dimensão da matriz documento-termo com TF-IDF: (5000, 5859)
Objeto vetorizador TF-IDF:


,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",<function Pre...t 0x11772b920>
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyz

In [12]:
y_pred_tfidf_raw, acc_tfidf_raw = run_train_predict_model(X_tfidf.toarray(), y, scaling='standard')

Acc = 0.178


### **2. Vetor semântico:**
<font size=3>

Agora, compare ambos os **vetores de frequência** com os **vetores semânticos**. Para isso, você irá capturar os vetores semânticos do tipo **conceito-documento** (já que estamos classificando documentos/resenhas) usando a função [`TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html) (ver **_Notebook_ 5**).

**Descreva** em uma célula *markdown* o resultado obtido.

In [13]:
def run_svd(X):
  svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=RANDOM_STATE)
  D_k = svd.fit_transform(X)
  return D_k, svd

#### BoW

In [14]:
D_k_bow, svd_bow = run_svd(X_bow.toarray())
print("Conceitos latentes dominantes (Uₖ) BoW:")
print(svd_bow.explained_variance_ratio_)
soma_variancia_bow = svd_bow.explained_variance_ratio_.sum()
print("Soma Total:", soma_variancia_bow)

print(f"Dimensão da projeção conceito-documento com BoW: {D_k_bow.shape}")

y_pred_bow_svd, acc_bow_svd = run_train_predict_model(D_k_bow, y, scaling='standard')

Conceitos latentes dominantes (Uₖ) BoW:
[0.04452358 0.01451407 0.01272572 0.01052187 0.00852935 0.00829551
 0.00741526 0.00736355 0.00684137 0.00649164 0.00630828 0.00591191
 0.00574377 0.00551424 0.00517205 0.00508418 0.00502903 0.00493165
 0.00478775 0.00450815 0.00444917 0.00430418 0.00423495 0.00412503
 0.00404103]
Soma Total: 0.20136729251750282
Dimensão da projeção conceito-documento com BoW: (5000, 25)
Acc = 0.298


#### TF-IDF

In [15]:
D_k_tfidf, svd_tfidf = run_svd(X_tfidf.toarray())

print("Conceitos latentes dominantes (Uₖ) TF-IDF:")
print(svd_tfidf.explained_variance_ratio_)
soma_variancia_tfidf = svd_tfidf.explained_variance_ratio_.sum()
print("Soma Total:", soma_variancia_tfidf)

print(f"Dimensão da projeção conceito-documento com TF-IDF: {D_k_tfidf.shape}")

y_pred_tfidf_svd, acc_tfidf_svd = run_train_predict_model(D_k_tfidf, y, scaling='standard')

Conceitos latentes dominantes (Uₖ) TF-IDF:
[0.0039163  0.00597419 0.00540807 0.00479838 0.00436174 0.00379424
 0.00362604 0.00340816 0.00313753 0.00309375 0.00296447 0.00285958
 0.00276519 0.00268957 0.00258925 0.00254078 0.00251195 0.00248098
 0.00233328 0.00223243 0.00218684 0.00215927 0.00209347 0.00203775
 0.00201763]
Soma Total: 0.07798080968793199
Dimensão da projeção conceito-documento com TF-IDF: (5000, 25)
Acc = 0.390


### **2.1. Comparação:**
<font size=3>

In [16]:
df_acc = pd.DataFrame(
    {
        "raw": [acc_bow_raw, acc_tfidf_raw],
        "SVD": [acc_bow_svd, acc_tfidf_svd],
        "Variância SVD (Σₖ)": [soma_variancia_bow, soma_variancia_tfidf],
    },
    index=["BoW", "TF-IDF"]
)
print("Parâmetros Gerais:")
print(f"MIN_DF = {MIN_DF}")
print(f"MAX_DF = {MAX_DF}")
print(f"TEST_SIZE = {TEST_SIZE}")
print(f"SVD_COMPONENTS = {SVD_COMPONENTS}")
print("Tabela de acurácias:")
display(df_acc)

Parâmetros Gerais:
MIN_DF = 5
MAX_DF = 0.95
TEST_SIZE = 0.1
SVD_COMPONENTS = 25
Tabela de acurácias:


,raw,SVD,Variância SVD (Σₖ)
BoW,0.220,0.298,0.201367
TF-IDF,0.178,0.390,0.077981


Os resultados de acurácia foram baixos em todos os experimentos. Ainda assim, observa-se um padrão consistente: as representações BoW e TF-IDF em sua forma original apresentaram desempenho inferior em relação às versões reduzidas via SVD. Esse comportamento é esperado, uma vez que as matrizes originais possuem alta dimensionalidade (≈5.000 atributos) e elevada esparsidade, o que prejudica algoritmos baseados em distância, como o k-NN.

Em espaços de alta dimensionalidade, as distâncias tornam-se menos discriminativas (“maldição da dimensionalidade”), comprometendo a noção de proximidade. Ao aplicar SVD e projetar os dados em um espaço de menor dimensionalidade (25 componentes), obtém-se uma representação densa em que os documentos são descritos em termos de conceitos latentes, enquanto os termos são associados a esses conceitos e seus pesos são determinados por Σₖ: 

$$A^T_{(doc-term)} \approx (U_k\cdot \Sigma_K\cdot V_k^T)^T = V_k\cdot \Sigma_k^T\cdot U_k^T $$

Nesse espaço, as distâncias passam a refletir melhor relações semânticas entre documentos, resultando em ganhos de desempenho.

Observa-se ainda que a diferença entre BoW e TF-IDF se amplia após o SVD: embora próximos no espaço original, o TF-IDF supera o BoW na representação reduzida (≈+10 p.p.). Isso sugere que a ponderação de termos do TF-IDF favorece a extração de conceitos latentes mais informativos.

Adicionalmente, o BoW apresenta maior contribuição dos conceitos latentes dominantes (~20%) em comparação ao TF-IDF (~7,8%), mas ainda assim obtém pior acurácia. Esse resultado indica que a maior relevância atribuída aos primeiros conceitos (capturada em Σₖ) não implica maior capacidade de discriminação: no BoW, parte dessa contribuição está associada a termos frequentes e pouco informativos, enquanto o TF-IDF direciona os conceitos para características mais relevantes.

Por fim, os experimentos com k-NN foram realizados com a configuração padrão (distância euclidiana). Considerando que, em tarefas de texto, a similaridade do cosseno é frequentemente utilizada por enfatizar a orientação dos vetores, investiga-se na sequência o impacto da substituição da métrica de distância sobre o desempenho do modelo.

### **3. Investigação exploratória: métrica de distância por cossenos (extra)**
<font size=3>

Em função da análise anterior, realiza-se uma nova rodada de experimentos utilizando a similaridade do cosseno como métrica no algoritmo k-NN. Nesta etapa, considera-se apenas a representação dos documentos projetada no espaço latente por meio do SVD, uma vez que os experimentos prévios indicaram desempenho inferior em espaços de alta dimensionalidade.

A função de treinamento utilizada permite a parametrização do modelo, sendo o k-NN com k=5 a configuração padrão. Adicionalmente, a função foi adaptada para suportar diferentes estratégias de pré-processamento. Em particular, substitui-se o uso do StandardScaler pela normalização dos vetores, uma vez que a similaridade do cosseno depende da orientação dos vetores no espaço e não de sua magnitude. O uso do StandardScaler pode alterar essa orientação, enquanto a normalização preserva a estrutura angular dos dados, tornando-se mais adequada nesse contexto.

In [17]:
cosine_bow_model=KNeighborsClassifier(n_neighbors=5, metric='cosine')
cosine_tfidf_model=KNeighborsClassifier(n_neighbors=5, metric='cosine')

y_pred_bow_cosine, acc_bow_cosine = run_train_predict_model(D_k_bow, y, cosine_bow_model, scaling='normalize')
y_pred_tfidf_cosine, acc_tfidf_cosine = run_train_predict_model(D_k_tfidf, y, cosine_tfidf_model, scaling='normalize')

df_acc_cosine = pd.DataFrame(
    {
        "Distancia Euclidiana": [acc_bow_svd, acc_tfidf_svd],
        "Distancia por Cosseno": [acc_bow_cosine, acc_tfidf_cosine],
    },
    index=["BoW", "TF-IDF"],
)

print("Tabela de acurácias:")
display(df_acc_cosine.style.format("{:.3f}"))


Acc = 0.370
Acc = 0.380
Tabela de acurácias:


,Distancia Euclidiana,Distancia por Cosseno
BoW,0.298,0.370
TF-IDF,0.390,0.380


Os resultados agora mostraram uma melhora significativa no BoW utilizando métrica de distancia por cossenos. Já para TF-IDF, houve leve piora no resultado. Intuitivamente esperava-se melhora em ambos casos. Na vetorização BoW + cossenos isso se confirmou, porém o resultado de TD-IDF + cossenos isso não ocorreu.

### **4. Conclusão**<font size=3>

Os resultados indicam que o impacto da métrica de distância no desempenho do k-NN depende da representação vetorial utilizada, mesmo após a redução de dimensionalidade por SVD. Para a representação baseada em BoW, o uso da similaridade do cosseno resultou em ganho expressivo, elevando a acurácia de 0.298 para 0.370. Esse comportamento sugere que, mesmo no espaço latente, a direção dos vetores permanece mais informativa do que sua magnitude nesse caso.

Por outro lado, para a representação baseada em TF-IDF, observou-se comportamento distinto: a distância euclidiana apresentou melhor desempenho (0.390) em comparação à similaridade do cosseno (0.380). Esse resultado indica que, após a aplicação do SVD, a magnitude dos componentes latentes passa a carregar informação discriminativa relevante, tornando a distância euclidiana mais adequada.

De forma geral, os resultados evidenciam que a escolha da métrica de distância e da representação dos dados deve ser feita de forma conjunta, uma vez que diferentes combinações podem levar a desempenhos significativamente distintos, mesmo quando se utilizam projeções em espaços semânticos latentes em tarefas de classificação com k-NN.

## __Referências:__
<font size=3>

- **(4.1 - 4.6; 6.1; 7.2)**: K. Faceli, _Inteligência artificial: uma abordagem de aprendizado de máquina_. Grupo Gen - LTC, 2011.
- **(15.0 - 15.6; 14.0 - 14.3; 17.0 - 17.4):** Gallatin, K. and Albon, C. *Machine Learning with Python Cookbook*, 2nd ed, 2023.
  